# User-Based Collaborative Filtering (80-20 Random Split)

## Split Strategy

This notebook uses **80-20 random split** for train/test division:
- **80%** of ratings randomly selected for training
- **20%** of ratings randomly selected for testing
- No temporal ordering considered

**Comparison with Temporal Split**:
- **Random split**: Evaluates how well the model generalizes to randomly held-out ratings
- **Temporal split**: Evaluates how well the model predicts future ratings (more realistic)

Random split typically yields better metrics since test data is similar to training data (no temporal drift).

## Algorithm Overview

**Concept**: Users with similar rating patterns in the past will likely have similar preferences in the future.

**Key Steps**:
1. Create user-item rating matrix
2. Apply mean-centering to remove user bias (some users rate higher than others)
3. Compute user-user similarity using Pearson correlation
4. For each user, predict ratings based on:
   - Weighted average of ratings from k most similar users
5. Add back user's mean rating (denormalization)

**Advantages**:
- Intuitive and easy to explain
- Can discover new genres/items from similar users

**Challenges**:
- Less stable than item-based (user preferences change)
- Scalability issues (more users than items)
- Cold-start for new users

**Expected Performance**: RMSE 0.88-0.98

## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Train/Test Data

In [ ]:
# Load 80-20 random split data
train_path = '../../datasets/output/split_and_train_datasets/80-20/train_ratings.csv'
test_path = '../../datasets/output/split_and_train_datasets/80-20/test_ratings.csv'

print("=" * 60)
print("LOADING 80-20 RANDOM SPLIT DATA")
print("=" * 60)
print("\nSplit strategy: 80% train / 20% test (random)")
print("  - Random shuffle of all ratings")
print("  - No temporal ordering")
print("  - Evaluates generalization to held-out ratings")
print()

print("Loading training data...")
train = pd.read_csv(train_path)
print(f"Train shape: {train.shape}")

print("\nLoading test data...")
test = pd.read_csv(test_path)
print(f"Test shape: {test.shape}")

print("\nData loaded successfully!")
print("=" * 60)

In [4]:
# Dataset statistics
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)

print("\nTraining Set:")
print(f"  Unique users: {train['userId'].nunique():,}")
print(f"  Unique movies: {train['movieId'].nunique():,}")
print(f"  Total ratings: {len(train):,}")
print(f"  Mean rating: {train['rating'].mean():.2f}")

print("\nTest Set:")
print(f"  Unique users: {test['userId'].nunique():,}")
print(f"  Unique movies: {test['movieId'].nunique():,}")
print(f"  Total ratings: {len(test):,}")
print(f"  Mean rating: {test['rating'].mean():.2f}")

DATASET STATISTICS

Training Set:
  Unique users: 227,222
  Unique movies: 25,636
  Total ratings: 20,819,431
  Mean rating: 3.52

Test Set:
  Unique users: 48,464
  Unique movies: 42,721
  Total ratings: 5,204,858
  Mean rating: 3.56


## 3. Create User-Item Matrix with Mean-Centering

In [5]:
# Create ID mappings
print("Creating ID mappings...")

unique_users = train['userId'].unique()
unique_movies = train['movieId'].unique()

user_id_map = {id: idx for idx, id in enumerate(unique_users)}
movie_id_map = {id: idx for idx, id in enumerate(unique_movies)}

idx_to_user = {idx: id for id, idx in user_id_map.items()}
idx_to_movie = {idx: id for id, idx in movie_id_map.items()}

print(f"Users: {len(user_id_map):,}")
print(f"Movies: {len(movie_id_map):,}")

Creating ID mappings...
Users: 227,222
Movies: 25,636


In [6]:
# Calculate per-user mean ratings (for mean-centering)
print("\nCalculating per-user mean ratings...")

user_mean_ratings = train.groupby('userId')['rating'].mean().to_dict()
global_mean_rating = train['rating'].mean()

print(f"Global mean rating: {global_mean_rating:.3f}")
print(f"User mean rating range: [{min(user_mean_ratings.values()):.2f}, {max(user_mean_ratings.values()):.2f}]")

# Apply mean-centering to training data
print("\nApplying mean-centering to training data...")
train['rating_centered'] = train.apply(
    lambda x: x['rating'] - user_mean_ratings.get(x['userId'], global_mean_rating),
    axis=1
)

print(f"Centered rating range: [{train['rating_centered'].min():.2f}, {train['rating_centered'].max():.2f}]")
print(f"Centered rating mean: {train['rating_centered'].mean():.4f} (should be ~0)")


Calculating per-user mean ratings...
Global mean rating: 3.521
User mean rating range: [0.50, 5.00]

Applying mean-centering to training data...
Centered rating range: [-4.39, 4.29]
Centered rating mean: 0.0000 (should be ~0)


In [7]:
# Map to matrix indices
print("\nMapping to matrix indices...")
train['user_idx'] = train['userId'].map(user_id_map)
train['movie_idx'] = train['movieId'].map(movie_id_map)

test['user_idx'] = test['userId'].map(user_id_map)
test['movie_idx'] = test['movieId'].map(movie_id_map)

# Check cold-start
cold_start_users = test['user_idx'].isna().sum()
cold_start_movies = test['movie_idx'].isna().sum()

print(f"Cold-start users: {cold_start_users:,} ({cold_start_users/len(test)*100:.1f}%)")
print(f"Cold-start movies: {cold_start_movies:,} ({cold_start_movies/len(test)*100:.1f}%)")


Mapping to matrix indices...
Cold-start users: 4,645,394 (89.3%)
Cold-start movies: 401,578 (7.7%)


In [ ]:
# Create sparse user-item matrices
print("\nCreating sparse user-item matrices...")

start_time = time.time()

# CSR format for row access (users)
user_item_matrix_csr = csr_matrix(
    (train['rating_centered'].values,
     (train['user_idx'].values, train['movie_idx'].values)),
    shape=(len(user_id_map), len(movie_id_map))
)

# CSC format for column access (movies) - MUCH FASTER for getting movie columns
print("Converting to CSC format for fast movie access...")
user_item_matrix_csc = user_item_matrix_csr.tocsc()

elapsed = time.time() - start_time

print(f"\nMatrices created in {elapsed:.2f} seconds")
print(f"Shape: {user_item_matrix_csr.shape}")
print(f"Memory (CSR): {user_item_matrix_csr.data.nbytes / (1024**2):.2f} MB")
print(f"Memory (CSC): {user_item_matrix_csc.data.nbytes / (1024**2):.2f} MB")
print(f"Total memory: {(user_item_matrix_csr.data.nbytes + user_item_matrix_csc.data.nbytes) / (1024**2):.2f} MB")

## 4. User-User Similarity - On-Demand Computation

**Approach**: We compute user-user similarities **on-demand during prediction** rather than pre-computing the full similarity matrix.

**Why?** 
- Full similarity matrix: 227,222 × 227,222 × 8 bytes = **412 GB** of RAM
- On-demand: Only compute similarities for relevant users per prediction

**How it works**: 
- For each prediction, find users who rated the target movie
- Compute similarities only between target user and those candidates
- Much more memory-efficient and scalable

In [ ]:
print("=" * 60)
print("PREPARING FOR ON-DEMAND SIMILARITY COMPUTATION")
print("=" * 60)
print("\nOptimizations for efficient prediction:")
print("  1. Pre-compute user norms in batches (reusable)")
print("  2. Use CSC matrix for fast movie column access")
print("  3. Limit candidates per prediction for scalability")
print("  4. Compute similarities on-demand (streaming)")
print("  5. Batch processing with periodic garbage collection")
print("\nPre-computing user norms in batches...")

import gc

# Pre-compute user norms in batches for efficiency
# Norms are computed once and reused for all predictions

n_users = user_item_matrix_csr.shape[0]
batch_size = 10000  # Process 10K users at a time
user_norms = np.zeros(n_users)

print(f"Processing {n_users:,} users in batches of {batch_size:,}...")

for batch_start in tqdm(range(0, n_users, batch_size), desc="Computing norms"):
    batch_end = min(batch_start + batch_size, n_users)
    
    # Get batch of users
    batch_matrix = user_item_matrix_csr[batch_start:batch_end]
    
    # Compute norms for this batch
    batch_norms = np.sqrt(batch_matrix.multiply(batch_matrix).sum(axis=1).A1)
    
    # Store results
    user_norms[batch_start:batch_end] = batch_norms
    
    # Clean up batch data
    del batch_matrix, batch_norms
    
    # Periodic garbage collection
    if (batch_start // batch_size) % 5 == 0:
        gc.collect()

# Final cleanup
gc.collect()

print(f"\n✓ Pre-computed {len(user_norms):,} user norms")
print(f"  Memory: {user_norms.nbytes / (1024**2):.2f} MB")
print("\nReady for prediction!")
print("=" * 60)

## 5. Prediction Function

In [ ]:
def predict_rating(user_idx, movie_idx, k=50, max_candidates=500):
    """
    User-Based CF prediction with on-demand similarity computation.
    
    This function computes similarities dynamically rather than using 
    a pre-computed similarity matrix, making it memory-efficient.
    
    Parameters:
    - user_idx: Index of user
    - movie_idx: Index of movie  
    - k: Number of similar users to consider (default: 50)
    - max_candidates: Max users to consider per prediction (default: 500)
    
    Returns:
    - predicted_rating: Float [0.5, 5.0]
    """
    # Handle cold-start cases
    if pd.isna(user_idx) or user_idx >= user_item_matrix_csr.shape[0]:
        return global_mean_rating
    
    if pd.isna(movie_idx) or movie_idx >= user_item_matrix_csc.shape[1]:
        user_id = idx_to_user.get(int(user_idx))
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    user_idx = int(user_idx)
    movie_idx = int(movie_idx)
    
    # Get users who rated this movie (using CSC for efficient column access)
    movie_col = user_item_matrix_csc[:, movie_idx]
    users_who_rated = movie_col.nonzero()[0]
    
    if len(users_who_rated) == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    # Remove target user from candidates
    users_who_rated = users_who_rated[users_who_rated != user_idx]
    
    if len(users_who_rated) == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    # Limit candidates for efficiency (prevents processing too many users)
    if len(users_who_rated) > max_candidates:
        users_who_rated = np.random.choice(users_who_rated, max_candidates, replace=False)
    
    # Get target user's rating vector
    target_user_row = user_item_matrix_csr[user_idx]
    target_norm = user_norms[user_idx]
    
    if target_norm == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    # Compute similarities on-demand (one user at a time)
    # This streaming approach is memory-efficient
    similarities = []
    ratings_centered = []
    
    for other_user_idx in users_who_rated:
        # Get candidate user's norm (pre-computed)
        other_norm = user_norms[other_user_idx]
        
        if other_norm == 0:
            continue
        
        # Compute cosine similarity between target and candidate user
        # Cosine on centered ratings approximates Pearson correlation
        dot_product = target_user_row.dot(user_item_matrix_csr[other_user_idx].T).toarray()[0, 0]
        
        similarity = dot_product / (target_norm * other_norm)
        
        # Get this user's centered rating for the movie
        rating_centered = movie_col[other_user_idx, 0]
        
        similarities.append(similarity)
        ratings_centered.append(rating_centered)
    
    if len(similarities) == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    # Convert to numpy arrays
    similarities = np.array(similarities)
    ratings_centered = np.array(ratings_centered)
    
    # Select top-k most similar users
    if len(similarities) > k:
        top_k_indices = np.argpartition(similarities, -k)[-k:]
        top_k_indices = top_k_indices[np.argsort(similarities[top_k_indices])[::-1]]
        
        top_k_sims = similarities[top_k_indices]
        top_k_ratings = ratings_centered[top_k_indices]
    else:
        top_k_sims = similarities
        top_k_ratings = ratings_centered
    
    # Filter out near-zero similarities
    valid = np.abs(top_k_sims) > 1e-6
    top_k_sims = top_k_sims[valid]
    top_k_ratings = top_k_ratings[valid]
    
    if len(top_k_sims) == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    # Compute weighted average of centered ratings
    weighted_sum = np.sum(top_k_sims * top_k_ratings)
    sum_of_weights = np.sum(np.abs(top_k_sims))
    
    if sum_of_weights == 0:
        user_id = idx_to_user[user_idx]
        return user_mean_ratings.get(user_id, global_mean_rating)
    
    predicted_centered = weighted_sum / sum_of_weights
    
    # Denormalize: add back user's mean rating
    user_id = idx_to_user[user_idx]
    user_mean = user_mean_ratings.get(user_id, global_mean_rating)
    predicted = predicted_centered + user_mean
    
    # Clip to valid rating range
    return np.clip(predicted, 0.5, 5.0)

print("✓ Prediction function defined!")
print("\nKey features:")
print("  ✓ On-demand similarity computation")
print("  ✓ Uses pre-computed norms for efficiency")
print("  ✓ Limits candidates to", 500, "users per prediction")
print("  ✓ Cosine on centered ratings (≈ Pearson correlation)")
print("  ✓ Mean-centering handles user rating bias")

## 6. Test on Sample

In [ ]:
# Test on samples
print("Testing prediction on 3 random samples...\n")
print("(Using small sample to verify function works before full run)")

# Use only non-cold-start users/movies for initial test
sample_test = test.dropna(subset=['user_idx', 'movie_idx']).sample(min(3, len(test)), random_state=42)

for idx, row in sample_test.iterrows():
    try:
        user_idx = int(row['user_idx'])
        movie_idx = int(row['movie_idx'])
        actual = row['rating']
        
        predicted = predict_rating(user_idx, movie_idx, k=50)
        
        print(f"User {row['userId']}, Movie {row['movieId']}")
        print(f"  Actual: {actual:.1f}, Predicted: {predicted:.2f}, Error: {abs(actual - predicted):.2f}")
        print()
    except Exception as e:
        print(f"ERROR on User {row['userId']}, Movie {row['movieId']}: {e}")
        print()

print("✓ Test completed successfully! Function is working.")
print("  You can now proceed to cell 18 for full evaluation.")

## 7. Generate Predictions for Test Set

In [ ]:
# Configuration for realistic evaluation
SAMPLE_SIZE = 100000  # 100K samples for fair comparison across algorithms
BATCH_SIZE = 1000     # Process 1000 predictions per batch
GC_FREQUENCY = 5      # Garbage collection every 5 batches

# Filter to only testable ratings (non-cold-start)
testable = test.dropna(subset=['user_idx', 'movie_idx'])

# IMPORTANT: Calculate TRUE coverage BEFORE sampling
true_coverage = len(testable) / len(test) * 100
testable_count_before_sampling = len(testable)

if SAMPLE_SIZE and SAMPLE_SIZE < len(testable):
    print(f"Using {SAMPLE_SIZE:,} random samples from test set")
    test_sample = testable.sample(SAMPLE_SIZE, random_state=42)
else:
    print(f"Using all testable ratings ({len(testable):,})")
    test_sample = testable.copy()

print(f"Total test ratings: {len(test):,}")
print(f"Testable ratings (user/movie in training): {testable_count_before_sampling:,}")
print(f"Coverage: {true_coverage:.2f}%")
print(f"Sampled for evaluation: {len(test_sample):,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"GC frequency: Every {GC_FREQUENCY} batches")
print(f"\nGenerating predictions for {len(test_sample):,} ratings...")
print("Estimated time: 15-30 minutes for 100K samples")
print("(Fair comparison sample size with other algorithms)")
print(f"Note: Coverage calculated on full test set, not sample\n")

# Memory monitoring
import psutil
mem_before = psutil.Process().memory_info().rss / (1024**3)
print(f"Memory before predictions: {mem_before:.2f} GB\n")

start_time = time.time()

# Batch processing with garbage collection
predictions = []
n_batches = (len(test_sample) + BATCH_SIZE - 1) // BATCH_SIZE

test_rows = test_sample[['user_idx', 'movie_idx']].values

for batch_idx in tqdm(range(n_batches), desc="Processing batches"):
    batch_start = batch_idx * BATCH_SIZE
    batch_end = min((batch_idx + 1) * BATCH_SIZE, len(test_rows))
    
    batch_predictions = []
    
    for i in range(batch_start, batch_end):
        user_idx, movie_idx = test_rows[i]
        pred = predict_rating(user_idx, movie_idx, k=50, max_candidates=500)
        batch_predictions.append(pred)
    
    predictions.extend(batch_predictions)
    
    # Periodic garbage collection
    if batch_idx % GC_FREQUENCY == 0:
        gc.collect()
        
        # Memory monitoring every 10 batches
        if batch_idx % 10 == 0 and batch_idx > 0:
            mem_current = psutil.Process().memory_info().rss / (1024**3)
            progress = len(predictions) / len(test_sample) * 100
            print(f"\n  Progress: {progress:.1f}% | Memory: {mem_current:.2f} GB")

# Final garbage collection
gc.collect()

elapsed = time.time() - start_time
mem_after = psutil.Process().memory_info().rss / (1024**3)

print(f"\n✓ Predictions completed in {elapsed/60:.2f} minutes")
print(f"  Average: {elapsed/len(test_sample)*1000:.2f} ms per rating")
print(f"  Memory used: {mem_after:.2f} GB (increased by {mem_after - mem_before:.2f} GB)")
print(f"\n✓ Ready for evaluation!")

# Performance suggestions
if elapsed/60 > 30:
    print("\n💡 TIP: To speed up, reduce max_candidates from 500 to 300")
elif mem_after > 10:
    print("\n⚠️  High memory usage. If issues occur, reduce SAMPLE_SIZE to 15000")

## 8. Evaluation Metrics

In [ ]:
# Calculate metrics
test_sample = test_sample.copy()
test_sample['predicted_rating'] = predictions

actual = test_sample['rating'].values
predicted = test_sample['predicted_rating'].values

rmse = np.sqrt(mean_squared_error(actual, predicted))
mae = mean_absolute_error(actual, predicted)

print("="*60)
print("USER-BASED COLLABORATIVE FILTERING RESULTS")
print("="*60)
print(f"\nAlgorithm: User-Based CF")
print(f"Similarity: Cosine on centered ratings (≈ Pearson)")
print(f"Normalization: Mean-centering per user")
print(f"Neighbors (k): 50")
print(f"Max candidates per prediction: 500")
print(f"Test samples: {len(test_sample):,}")
print(f"\nRMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"\nPrediction time: {elapsed/60:.2f} minutes")
print(f"Per rating: {elapsed/len(test_sample)*1000:.2f} ms")

# Coverage (use saved true coverage)
print(f"\nCoverage: {true_coverage:.2f}%")

# Memory summary
print("\n" + "="*60)
print("MEMORY USAGE SUMMARY")
print("="*60)
print(f"CSR Matrix: {user_item_matrix_csr.data.nbytes / (1024**2):.2f} MB")
print(f"CSC Matrix: {user_item_matrix_csc.data.nbytes / (1024**2):.2f} MB")
print(f"User norms: {user_norms.nbytes / (1024**2):.2f} MB")
total_mb = (user_item_matrix_csr.data.nbytes + user_item_matrix_csc.data.nbytes + user_norms.nbytes) / (1024**2)
print(f"Peak process memory: {mem_after:.2f} GB")
print(f"Base data structures: ~{total_mb:.2f} MB")
print("\n✓ On-demand similarity computation used")
print("="*60)

# Statistical significance
print("\n" + "="*60)
print("EVALUATION CONFIDENCE")
print("="*60)
print(f"Sample size: {len(test_sample):,} ratings")
print(f"Percentage of testable data: {len(test_sample)/testable_count_before_sampling*100:.2f}%")

# 95% confidence interval for RMSE
from scipy import stats
squared_errors = (actual - predicted) ** 2
se = stats.sem(squared_errors)
ci = se * stats.t.ppf((1 + 0.95) / 2., len(squared_errors)-1)
rmse_ci = np.sqrt(ci)

print(f"\nRMSE 95% CI: {rmse:.4f} ± {rmse_ci:.4f}")
print(f"Range: [{rmse - rmse_ci:.4f}, {rmse + rmse_ci:.4f}]")

if len(test_sample) >= 20000:
    print("\n✓ Sample size adequate for reliable evaluation")
else:
    print(f"\n⚠️  Consider increasing to 20K+ for more confidence")

print("="*60)

## 9. Visualizations

In [ ]:
# Plot 1: Actual vs Predicted
plt.figure(figsize=(10, 6))
plt.scatter(actual, predicted, alpha=0.3, s=1)
plt.plot([0.5, 5], [0.5, 5], 'r--', label='Perfect Prediction')
plt.xlabel('Actual Rating')
plt.ylabel('Predicted Rating')
plt.title('User-Based CF: Actual vs Predicted Ratings')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Correlation: {np.corrcoef(actual, predicted)[0, 1]:.4f}")

In [ ]:
# Plot 2: Error Distribution
errors = actual - predicted

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(errors, bins=50, edgecolor='black')
plt.xlabel('Prediction Error')
plt.ylabel('Frequency')
plt.title('Error Distribution')
plt.axvline(0, color='red', linestyle='--')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(errors)
plt.ylabel('Prediction Error')
plt.title('Error Boxplot')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean Error: {np.mean(errors):.4f}")
print(f"Std Error: {np.std(errors):.4f}")

## 10. Save Results

In [ ]:
# Save results
results = {
    'algorithm': 'User-Based CF (80-20 split)',
    'split_strategy': '80-20 random',
    'similarity_metric': 'Cosine (on centered ratings)',
    'normalization': 'Mean-centering per user',
    'k_neighbors': 50,
    'max_candidates': 200,
    'approach': 'Streaming (one-at-a-time)',
    'rmse': rmse,
    'mae': mae,
    'coverage': true_coverage,
    'training_time_minutes': elapsed/60,
    'prediction_time_ms': elapsed/len(test_sample)*1000,
    'test_samples': len(test_sample),
    'total_testable': testable_count_before_sampling
}

results_df = pd.DataFrame([results])
output_path = '../../datasets/output/model_implementations/user_based_cf_results_80_20.csv'
results_df.to_csv(output_path, index=False)

print(f"✓ Results saved to: {output_path}")
print("\nResults Summary:")
print(results_df.T)

## Summary

**User-Based Collaborative Filtering** implemented successfully!

**Key Features**:
- Mean-centering to handle user rating bias
- Cosine similarity on centered ratings (approximates Pearson correlation)
- k=50 nearest neighbors
- Handles cold-start with user/global means

**Next Steps**:
1. Implement SVD Matrix Factorization
2. Compare all three algorithms